# Hybrid Model Re-evaluation After GBM Leakage Correction

## 1. Objective

This notebook re-evaluates the Hybrid model developed in T39 by combining Chronos-Bolt predictions with the corrected GBM predictions produced in `02_GBM_Leakage_Correction.ipynb`.

The objective is to investigate whether the Hybrid model maintains its forecasting performance after correcting the GBM input features.

## 2. Original Hybrid Method

The original Hybrid combines Chronos and GBM predictions using a linear model with learned weights and an intercept.

The weights are estimated separately for each energy series using least squares.

The evaluation follows the original T39 protocol:

- **FIT:** 1 July – 15 October 2022. Used to estimate the Hybrid weights.
- **SEL:** 16 October – 31 December 2022. Used for out-of-sample model comparison.
- **TEST:** 2023 and 2025. Used for additional evaluation.

## 3. Methodology

1. Load the corrected GBM and original Chronos predictions.
2. Align predictions using UTC timestamps, zones and load types.
3. Verify that actual energy consumption values match.
4. Retrain the Hybrid weights using FIT observations, evaluate the resulting model on SEL, then re-estimate the weights on the full validation period (`w_full`) and apply these updated weights only to the 2023 and 2025 test periods.
5. Evaluate the retrained Hybrid on SEL, test 2023 and test 2025.
6. Export predictions and learned weights for the final model-selection audit.

## 4. Scope and Limitations

The original Hybrid method will be preserved, while its GBM input will be replaced with corrected GBM predictions.

The corrected GBM represents a fixed 24-hour forecasting-horizon experiment. It has not yet been validated as a complete operational next-day forecasting system.

This notebook provides an experimental re-evaluation and does not overwrite the official T39 results.

**References:** `hybrid_blend_learned_weights.ipynb` and `02_GBM_Leakage_Correction.ipynb`.

In [ ]:

from google.colab import files
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

uploaded = files.upload()

print("\nUploaded files:")
for name in uploaded:
    print(name)


Saving GBM_Corrected_Predictions_10series (1).csv to GBM_Corrected_Predictions_10series (1) (1).csv
Saving 2026G3-main.zip to 2026G3-main (1).zip

Uploaded files:
GBM_Corrected_Predictions_10series (1) (1).csv
2026G3-main (1).zip


In [ ]:

from pathlib import Path

print("CSV disponibili in Colab:")

for p in sorted(Path("/content").glob("*.csv")):
    print("-", p.name)


CSV disponibili in Colab:
- GBM_Corrected_Predictions_10series (1) (1).csv
- GBM_Corrected_Predictions_10series (1).csv


In [ ]:

# STEP 3 - Load repository and corrected GBM (10 series)

from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

zip_file = Path("/content/2026G3-main.zip")


gbm_files = sorted(
    Path("/content").glob(
        "GBM_Corrected_Predictions_10series*.csv"
    )
)

assert gbm_files, "Corrected GBM CSV not uploaded"

gbm_file = gbm_files[0]

print("Using GBM file:", gbm_file.name)


assert zip_file.exists(), "Repository ZIP not found"
assert gbm_file.exists(), "10-series corrected GBM CSV not found"

# Extract original repository
extract_dir = Path("/content/tbda_hybrid")
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_file) as z:
    z.extractall(extract_dir)

project_dir = extract_dir / "2026G3-main"

assert project_dir.exists(), "Project folder not found"

# Load corrected GBM in standard T38 format
corrected_gbm = pd.read_csv(gbm_file)

corrected_gbm["timestamp_utc"] = pd.to_datetime(
    corrected_gbm["timestamp_utc"],
    utc=True
)

# Convert columns to the names expected by Notebook 03
corrected_gbm = corrected_gbm.rename(columns={
    "y_true": "energy_wh_model",
    "y_pred": "pred_corrected"
})

print("Repository loaded:", project_dir)
print("GBM file:", gbm_file.name)
print("Corrected predictions:", len(corrected_gbm))

print("\nEnergy series:")
display(
    corrected_gbm.groupby(["zone", "load_type"])
    .size().reset_index(name="rows")
)

assert len(corrected_gbm) == 46007
assert corrected_gbm.groupby(
    ["zone", "load_type"]
).ngroups == 10

print("\n10-series GBM loaded successfully!")


Using GBM file: GBM_Corrected_Predictions_10series (1) (1).csv
Repository loaded: /content/tbda_hybrid/2026G3-main
GBM file: GBM_Corrected_Predictions_10series (1) (1).csv
Corrected predictions: 46007

Energy series:


,zone,load_type,rows
0,NE,HVAC,6234
1,NE,Plugs & Sockets,6203
2,NW,HVAC,5466
3,NW,Lighting,6207
4,NW,Plugs & Sockets,6204
5,SE,HVAC,3603
6,SE,Lighting,3679
7,SE,Plugs & Sockets,4585
8,SW,Lighting,1762
9,SW,Plugs & Sockets,2064



10-series GBM loaded successfully!


In [ ]:

# STEP 4 - Basic validation

required_columns = [
    "timestamp_utc",
    "zone",
    "load_type",
    "split",
    "energy_wh_model",
    "pred_corrected"
]

missing = [
    c for c in required_columns
    if c not in corrected_gbm.columns
]

assert not missing, f"Missing columns: {missing}"

duplicates = corrected_gbm.duplicated(
    ["timestamp_utc", "zone", "load_type"]
).sum()

missing_predictions = corrected_gbm["pred_corrected"].isna().sum()

print("GBM prediction rows:", len(corrected_gbm))
print("Duplicate predictions:", duplicates)
print("Missing predictions:", missing_predictions)

print("\nAvailable splits:")
print(corrected_gbm["split"].value_counts())

assert duplicates == 0, "Duplicate predictions detected"
assert missing_predictions == 0, "Missing GBM predictions detected"

print("\nBasic validation PASSED")


GBM prediction rows: 46007
Duplicate predictions: 0
Missing predictions: 0

Available splits:
split
validation    29524
test_2023      9394
test_2025      7089
Name: count, dtype: int64

Basic validation PASSED


In [ ]:

# STEP 5 - Find Chronos prediction archives

archive_dir = project_dir / "archive"

chronos_zips = sorted(
    p for p in archive_dir.glob("*.zip")
    if "chronos" in p.name.lower()
)

print("Chronos archives found:", len(chronos_zips))

for path in chronos_zips:
    print("-", path.name)

assert chronos_zips, "No Chronos archives found"


Chronos archives found: 4
- NE_Chronos.zip
- NW_Chronos.zip
- SE_Chronos.zip
- SW_Chronos.zip


In [ ]:

# STEP 6 - Inspect Chronos prediction files

from io import BytesIO

chronos_inventory = []
chronos_samples = []

for zip_path in chronos_zips:
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            if (
                name.lower().endswith("predictions.csv")
                and "__macosx" not in name.lower()
            ):
                data = pd.read_csv(BytesIO(z.read(name)))

                chronos_inventory.append({
                    "archive": zip_path.name,
                    "file": name,
                    "rows": len(data),
                    "columns": ", ".join(data.columns)
                })

                chronos_samples.append((name, data))

inventory_chronos = pd.DataFrame(chronos_inventory)

display(inventory_chronos)

print("\nExample prediction file:")
if chronos_samples:
    print(chronos_samples[0][0])
    display(chronos_samples[0][1].head(3))
else:
    print("No prediction files found")


,archive,file,rows,columns
0,NE_Chronos.zip,NE_Chronos/NE_Plugs_Sockets_Chronos/NE_Plugs_S...,6679,"timestamp_utc, zone, load_type, model, split, ..."
1,NE_Chronos.zip,NE_Chronos/NE_HVAC_Chronos/NE_HVAC_chronos_bol...,6714,"timestamp_utc, zone, load_type, model, split, ..."
2,NW_Chronos.zip,NW_Chronos/NW_RACK_Chronos/NW_RACK_chronos_bol...,6670,"timestamp_utc, zone, load_type, model, split, ..."
3,NW_Chronos.zip,NW_Chronos/NW_Plugs&Socket_Chronos/NW_Plugs_So...,6713,"timestamp_utc, zone, load_type, model, split, ..."
4,NW_Chronos.zip,NW_Chronos/NW_Emerg_Lightning_Chronos/NW_Emerg...,6075,"timestamp_utc, zone, load_type, model, split, ..."
5,NW_Chronos.zip,NW_Chronos/NW_HVAC_Chronos/NW_HVAC_chronos_bol...,6053,"timestamp_utc, zone, load_type, model, split, ..."
6,NW_Chronos.zip,NW_Chronos/NW_Lighting_Chronos/NW_Lighting_chr...,6712,"timestamp_utc, zone, load_type, model, split, ..."
7,SE_Chronos.zip,SE_Chronos/SE_Plugs&Sockets_Chronos/SE_Plugs_S...,6157,"timestamp_utc, zone, load_type, model, split, ..."
8,SE_Chronos.zip,SE_Chronos/SE_Emerg_Lightning_Chronos/SE_Emerg...,5956,"timestamp_utc, zone, load_type, model, split, ..."
9,SE_Chronos.zip,SE_Chronos/SE_HVAC_Chronos/SE_HVAC_chronos_bol...,5949,"timestamp_utc, zone, load_type, model, split, ..."



Example prediction file:
NE_Chronos/NE_Plugs_Sockets_Chronos/NE_Plugs_Sockets_chronos_bolt_base_predictions.csv


,timestamp_utc,zone,load_type,model,split,y_true,y_pred
0,2022-07-01 00:00:00,NE,Plugs & Sockets,chronos_bolt_base,validation,188.573,188.392105
1,2022-07-01 01:00:00,NE,Plugs & Sockets,chronos_bolt_base,validation,187.774,188.035385
2,2022-07-01 02:00:00,NE,Plugs & Sockets,chronos_bolt_base,validation,187.628,187.795273


In [ ]:

# STEP 7 - Combine Chronos predictions

chronos_frames = []

for name, data in chronos_samples:
    required = {
        "timestamp_utc",
        "zone",
        "load_type",
        "model",
        "y_true",
        "y_pred"
    }

    if not required.issubset(data.columns):
        print("Skipped (missing columns):", name)
        continue

    chronos_frames.append(data.copy())

assert chronos_frames, "No compatible Chronos files found"

chronos_all = pd.concat(
    chronos_frames,
    ignore_index=True
)

chronos_all["timestamp_utc"] = pd.to_datetime(
    chronos_all["timestamp_utc"],
    utc=True
)

print("Chronos rows:", len(chronos_all))

print("\nModels:")
print(chronos_all["model"].value_counts())

print("\nDuplicate keys:")
print(
    chronos_all.duplicated(
        ["timestamp_utc", "zone", "load_type", "model"]
    ).sum()
)


Chronos rows: 82731

Models:
model
chronos_bolt_base    82731
Name: count, dtype: int64

Duplicate keys:
0


In [ ]:

# STEP 8 - Preliminary overlap check

chronos_models = chronos_all["model"].unique()

print("Available Chronos models:")
print(chronos_models)

print("\nCorrected GBM series:")
display(
    corrected_gbm.groupby(
        ["zone", "load_type"]
    ).size().reset_index(name="gbm_rows")
)

print("\nChronos series:")
display(
    chronos_all.groupby(
        ["zone", "load_type", "model"]
    ).size().reset_index(name="chronos_rows")
)


Available Chronos models:
['chronos_bolt_base']

Corrected GBM series:


,zone,load_type,gbm_rows
0,NE,HVAC,6234
1,NE,Plugs & Sockets,6203
2,NW,HVAC,5466
3,NW,Lighting,6207
4,NW,Plugs & Sockets,6204
5,SE,HVAC,3603
6,SE,Lighting,3679
7,SE,Plugs & Sockets,4585
8,SW,Lighting,1762
9,SW,Plugs & Sockets,2064



Chronos series:


,zone,load_type,model,chronos_rows
0,NE,HVAC,chronos_bolt_base,6714
1,NE,Plugs & Sockets,chronos_bolt_base,6679
2,NW,Emerg. Lighting,chronos_bolt_base,6075
3,NW,HVAC,chronos_bolt_base,6053
4,NW,Lighting,chronos_bolt_base,6712
5,NW,Plugs & Sockets,chronos_bolt_base,6713
6,NW,RACK,chronos_bolt_base,6670
7,SE,Emerg. Lighting,chronos_bolt_base,5956
8,SE,HVAC,chronos_bolt_base,5949
9,SE,Lighting,chronos_bolt_base,5903


In [ ]:

# STEP 9 - Align corrected GBM and Chronos predictions

gbm_part = corrected_gbm[
    ["timestamp_utc", "zone", "load_type",
     "energy_wh_model", "pred_corrected"]
].copy()

chronos_part = chronos_all[
    ["timestamp_utc", "zone", "load_type",
     "y_true", "y_pred"]
].copy()

chronos_part = chronos_part.rename(columns={
    "y_true": "chronos_actual",
    "y_pred": "pred_chronos"
})

hybrid_data = gbm_part.merge(
    chronos_part,
    on=["timestamp_utc", "zone", "load_type"],
    how="inner",
    validate="one_to_one"
)

hybrid_data["actual_difference"] = (
    hybrid_data["energy_wh_model"]
    - hybrid_data["chronos_actual"]
).abs()

print("Aligned observations:", len(hybrid_data))

print("\nActual value differences:")
print("Maximum difference:",
      hybrid_data["actual_difference"].max())

print("Rows with difference > 0.01:",
      (hybrid_data["actual_difference"] > 0.01).sum())

print("\nMatched series:")
display(
    hybrid_data.groupby(["zone", "load_type"])
    .size().reset_index(name="matched_hours")
)


Aligned observations: 46007

Actual value differences:
Maximum difference: 0.0
Rows with difference > 0.01: 0

Matched series:


,zone,load_type,matched_hours
0,NE,HVAC,6234
1,NE,Plugs & Sockets,6203
2,NW,HVAC,5466
3,NW,Lighting,6207
4,NW,Plugs & Sockets,6204
5,SE,HVAC,3603
6,SE,Lighting,3679
7,SE,Plugs & Sockets,4585
8,SW,Lighting,1762
9,SW,Plugs & Sockets,2064


In [ ]:

# STEP 10 - Define original Hybrid periods

fit_start = pd.Timestamp("2022-07-01", tz="UTC")
sel_start = pd.Timestamp("2022-10-16", tz="UTC")
sel_end = pd.Timestamp("2023-01-01", tz="UTC")

hybrid_data["hybrid_period"] = "OTHER"

hybrid_data.loc[
    (hybrid_data["timestamp_utc"] >= fit_start) &
    (hybrid_data["timestamp_utc"] < sel_start),
    "hybrid_period"
] = "FIT"

hybrid_data.loc[
    (hybrid_data["timestamp_utc"] >= sel_start) &
    (hybrid_data["timestamp_utc"] < sel_end),
    "hybrid_period"
] = "SEL"

print("Available observations by period:")

display(
    hybrid_data.groupby(
        ["zone", "load_type", "hybrid_period"]
    ).size().unstack(fill_value=0)
)


Available observations by period:


hybrid_period          FIT  OTHER   SEL
zone load_type                         
NE   HVAC             2057   2367  1810
     Plugs & Sockets  2057   2336  1810
NW   HVAC             2057   1806  1603
     Lighting         2057   2340  1810
     Plugs & Sockets  2057   2337  1810
SE   HVAC             1117   1378  1108
     Lighting         1200   1441  1038
     Plugs & Sockets  1724   1624  1237
SW   Lighting          837    371   554
     Plugs & Sockets   918    483   663

In [ ]:

# STEP 11 - Check training data availability

fit_counts = (
    hybrid_data[
        hybrid_data["hybrid_period"] == "FIT"
    ]
    .groupby(["zone", "load_type"])
    .size()
    .rename("fit_hours")
)

sel_counts = (
    hybrid_data[
        hybrid_data["hybrid_period"] == "SEL"
    ]
    .groupby(["zone", "load_type"])
    .size()
    .rename("sel_hours")
)

period_check = pd.concat(
    [fit_counts, sel_counts], axis=1
).fillna(0).astype(int)

display(period_check)

print("\nSeries with FIT and SEL data:",
      ((period_check["fit_hours"] >= 3) &
       (period_check["sel_hours"] > 0)).sum())

print("\nActual values aligned:",
      bool(
          (hybrid_data["actual_difference"] <= 0.01).all()
      ))


fit_hours  sel_hours
zone load_type                            
NE   HVAC                  2057       1810
     Plugs & Sockets       2057       1810
NW   HVAC                  2057       1603
     Lighting              2057       1810
     Plugs & Sockets       2057       1810
SE   HVAC                  1117       1108
     Lighting              1200       1038
     Plugs & Sockets       1724       1237
SW   Lighting               837        554
     Plugs & Sockets        918        663


Series with FIT and SEL data: 10

Actual values aligned: True


In [ ]:

# STEP 12 - Hybrid weights: FIT for SEL, FULL for tests

hybrid_weights = []
hybrid_predictions = []

for (zone, load_type), group in hybrid_data.groupby(
    ["zone", "load_type"]
):
    group = group.copy()

    fit = group[
        group["hybrid_period"] == "FIT"
    ].copy()

    full = group[
        group["hybrid_period"].isin(["FIT", "SEL"])
    ].copy()

    def prepare_xy(data):
        X = np.column_stack([
            data["pred_chronos"].to_numpy(dtype=float),
            data["pred_corrected"].to_numpy(dtype=float),
            np.ones(len(data))
        ])
        y = data["energy_wh_model"].to_numpy(dtype=float)

        assert np.isfinite(X).all()
        assert np.isfinite(y).all()

        return X, y

    # 1. Estimate weights on FIT only
    X_fit, y_fit = prepare_xy(fit)

    w_fit, _, rank_fit, _ = np.linalg.lstsq(
        X_fit, y_fit, rcond=None
    )

    assert rank_fit == 3, (
        f"FIT rank problem: {zone} {load_type}"
    )

    # 2. Re-estimate weights on full validation
    X_full, y_full = prepare_xy(full)

    w_full, _, rank_full, _ = np.linalg.lstsq(
        X_full, y_full, rcond=None
    )

    assert rank_full == 3, (
        f"FULL rank problem: {zone} {load_type}"
    )

    # 3. Generate both sets of predictions
    X_all, _ = prepare_xy(group)

    group["pred_hybrid_fit"] = X_all @ w_fit
    group["pred_hybrid_full"] = X_all @ w_full

    # 4. Use FIT weights for validation;
    # use FULL weights only for future tests
    group["pred_hybrid_corrected"] = np.where(
        group["hybrid_period"].isin(["FIT", "SEL"]),
        group["pred_hybrid_fit"],
        group["pred_hybrid_full"]
    )

    hybrid_weights.append({
        "zone": zone,
        "load_type": load_type,
        "n_fit": len(fit),
        "n_full": len(full),
        "w_chronos_fit": w_fit[0],
        "w_gbm_fit": w_fit[1],
        "intercept_fit": w_fit[2],
        "w_chronos_full": w_full[0],
        "w_gbm_full": w_full[1],
        "intercept_full": w_full[2]
    })

    hybrid_predictions.append(group)

weights_df = pd.DataFrame(hybrid_weights)

hybrid_corrected = pd.concat(
    hybrid_predictions,
    ignore_index=True
)

print("Hybrid training completed!")
print("Series trained:", len(weights_df))
print("Predictions generated:", len(hybrid_corrected))

assert len(weights_df) == 10

display(weights_df.round(4))


Hybrid training completed!
Series trained: 10
Predictions generated: 46007


,zone,load_type,n_fit,n_full,w_chronos_fit,w_gbm_fit,intercept_fit,w_chronos_full,w_gbm_full,intercept_full
0,NE,HVAC,2057,3867,0.7386,0.2466,0.0579,1.3635,-0.5580,0.7402
1,NE,Plugs & Sockets,2057,3867,0.8792,0.0460,13.9486,0.8513,0.1342,-0.6032
2,NW,HVAC,2057,3660,0.5148,0.4155,13.4887,0.5289,0.4381,3.6516
3,NW,Lighting,2057,3867,0.8227,0.0721,11.0109,0.8110,0.0706,12.1854
4,NW,Plugs & Sockets,2057,3867,0.4942,0.9819,-255.4545,0.4715,0.5862,16.1628
5,SE,HVAC,1117,2225,0.4417,0.5731,2.1939,0.7802,0.1027,9.7316
6,SE,Lighting,1200,2238,0.6308,0.2348,13.0204,1.1499,0.6928,14.8070
7,SE,Plugs & Sockets,1724,2961,0.9723,-0.0865,13.6851,1.3343,0.4519,-76.8595
8,SW,Lighting,837,1391,0.9433,1.8680,0.4945,0.7930,0.7734,12.8523
9,SW,Plugs & Sockets,918,1581,0.7684,0.1923,8.7347,0.7811,0.2553,-7.2226


In [ ]:

# Verify FIT and FULL Hybrid weight estimation

print("Series:", len(weights_df))

print("\nTraining observations:")
display(
    weights_df[
        ["zone", "load_type", "n_fit", "n_full"]
    ]
)

print("\nWeight differences:")
weights_df["weight_difference"] = (
    (weights_df["w_chronos_full"] -
     weights_df["w_chronos_fit"]).abs()
    +
    (weights_df["w_gbm_full"] -
     weights_df["w_gbm_fit"]).abs()
)

display(
    weights_df[
        ["zone", "load_type", "weight_difference"]
    ].round(6)
)

assert (weights_df["n_full"] >= weights_df["n_fit"]).all()

print("\nFIT and FULL weight verification PASSED")


Series: 10

Training observations:


,zone,load_type,n_fit,n_full
0,NE,HVAC,2057,3867
1,NE,Plugs & Sockets,2057,3867
2,NW,HVAC,2057,3660
3,NW,Lighting,2057,3867
4,NW,Plugs & Sockets,2057,3867
5,SE,HVAC,1117,2225
6,SE,Lighting,1200,2238
7,SE,Plugs & Sockets,1724,2961
8,SW,Lighting,837,1391
9,SW,Plugs & Sockets,918,1581



Weight differences:


,zone,load_type,weight_difference
0,NE,HVAC,1.429499
1,NE,Plugs & Sockets,0.116151
2,NW,HVAC,0.036681
3,NW,Lighting,0.013165
4,NW,Plugs & Sockets,0.418359
5,SE,HVAC,0.808870
6,SE,Lighting,0.977108
7,SE,Plugs & Sockets,0.900361
8,SW,Lighting,1.244937
9,SW,Plugs & Sockets,0.075692



FIT and FULL weight verification PASSED


In [ ]:

# STEP 13 - Compare models on SEL using FIT weights

import numpy as np
import pandas as pd

def cv_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0 or np.mean(y_true) <= 0:
        return np.nan

    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    return 100 * rmse / np.mean(y_true)

sel_data = hybrid_corrected[
    hybrid_corrected["hybrid_period"] == "SEL"
].copy()

results = []

for (zone, load_type), group in sel_data.groupby(
    ["zone", "load_type"]
):
    y = group["energy_wh_model"]

    scores = {
        "Chronos": cv_rmse(y, group["pred_chronos"]),
        "Corrected_GBM": cv_rmse(
            y, group["pred_corrected"]
        ),
        "Corrected_Hybrid": cv_rmse(
            y, group["pred_hybrid_fit"]
        )
    }

    valid_scores = {
        k: v for k, v in scores.items()
        if np.isfinite(v)
    }

    results.append({
        "zone": zone,
        "load_type": load_type,
        "n_hours": len(group),
        "Chronos_CV": scores["Chronos"],
        "Corrected_GBM_CV": scores["Corrected_GBM"],
        "Corrected_Hybrid_CV": scores["Corrected_Hybrid"],
        "Best_among_three": min(
            valid_scores, key=valid_scores.get
        ) if valid_scores else "N/A"
    })

hybrid_results = pd.DataFrame(results)

assert len(hybrid_results) == 10

print("SEL 2022 - CV(RMSE) %")
display(hybrid_results.round(2))

print("\nBest among three on SEL:")
print(hybrid_results["Best_among_three"].value_counts())


SEL 2022 - CV(RMSE) %


,zone,load_type,n_hours,Chronos_CV,Corrected_GBM_CV,Corrected_Hybrid_CV,Best_among_three
0,NE,HVAC,1810,326.62,326.76,326.63,Chronos
1,NE,Plugs & Sockets,1810,42.89,76.21,41.91,Corrected_Hybrid
2,NW,HVAC,1603,19.10,33.71,42.71,Chronos
3,NW,Lighting,1810,66.14,100.70,64.39,Corrected_Hybrid
4,NW,Plugs & Sockets,1810,49.25,50.06,56.68,Chronos
5,SE,HVAC,1108,23.24,50.28,34.90,Chronos
6,SE,Lighting,1038,211.59,272.22,225.58,Chronos
7,SE,Plugs & Sockets,1237,131.50,151.06,134.12,Chronos
8,SW,Lighting,554,302.14,324.42,318.35,Chronos
9,SW,Plugs & Sockets,663,12.12,14.42,12.44,Chronos



Best among three on SEL:
Best_among_three
Chronos             8
Corrected_Hybrid    2
Name: count, dtype: int64


In [ ]:

# STEP 14 - Evaluate test periods using FULL weights

test_periods = {
    "test_2023": (
        pd.Timestamp("2023-01-01", tz="UTC"),
        pd.Timestamp("2023-03-01", tz="UTC")
    ),
    "test_2025": (
        pd.Timestamp("2025-08-18", tz="UTC"),
        pd.Timestamp("2025-10-14", tz="UTC")
    )
}

test_results = []

for period, (start, end) in test_periods.items():

    period_data = hybrid_corrected[
        (hybrid_corrected["timestamp_utc"] >= start) &
        (hybrid_corrected["timestamp_utc"] < end)
    ].copy()

    for (zone, load_type), group in period_data.groupby(
        ["zone", "load_type"]
    ):
        y = group["energy_wh_model"]

        scores = {
            "Chronos": cv_rmse(
                y, group["pred_chronos"]
            ),
            "Corrected_GBM": cv_rmse(
                y, group["pred_corrected"]
            ),
            "Corrected_Hybrid": cv_rmse(
                y, group["pred_hybrid_full"]
            )
        }

        valid_scores = {
            k: v for k, v in scores.items()
            if np.isfinite(v)
        }

        test_results.append({
            "period": period,
            "zone": zone,
            "load_type": load_type,
            "n_hours": len(group),
            "Chronos_CV": scores["Chronos"],
            "Corrected_GBM_CV": scores["Corrected_GBM"],
            "Corrected_Hybrid_CV": scores["Corrected_Hybrid"],
            "Best_among_three": min(
                valid_scores, key=valid_scores.get
            ) if valid_scores else "N/A"
        })

test_results_df = pd.DataFrame(test_results)

print("TEST RESULTS - FULL WEIGHTS - CV(RMSE) %")
display(test_results_df.round(2))

print("\nDiagnostic best-model counts:")
display(
    test_results_df.groupby(
        ["period", "Best_among_three"]
    ).size().reset_index(name="series")
)

print("\nSeries evaluated per test:")
print(test_results_df.groupby("period").size())


TEST RESULTS - FULL WEIGHTS - CV(RMSE) %


,period,zone,load_type,n_hours,Chronos_CV,Corrected_GBM_CV,Corrected_Hybrid_CV,Best_among_three
0,test_2023,NE,HVAC,1318,686.56,686.89,686.29,Corrected_Hybrid
1,test_2023,NE,Plugs & Sockets,1318,20.54,54.15,20.98,Chronos
2,test_2023,NW,HVAC,802,35.13,48.52,37.82,Chronos
3,test_2023,NW,Lighting,1289,61.50,99.37,61.65,Chronos
4,test_2023,NW,Plugs & Sockets,1318,55.91,55.35,51.43,Corrected_Hybrid
5,test_2023,SE,HVAC,654,45.08,58.33,44.86,Corrected_Hybrid
6,test_2023,SE,Lighting,793,143.55,187.83,159.23,Chronos
7,test_2023,SE,Plugs & Sockets,1048,123.95,144.61,133.71,Chronos
8,test_2023,SW,Lighting,371,176.51,237.81,171.56,Corrected_Hybrid
9,test_2023,SW,Plugs & Sockets,483,14.42,29.14,15.37,Chronos



Diagnostic best-model counts:


,period,Best_among_three,series
0,test_2023,Chronos,6
1,test_2023,Corrected_Hybrid,4
2,test_2025,Chronos,5
3,test_2025,Corrected_GBM,1
4,test_2025,Corrected_Hybrid,2



Series evaluated per test:
period
test_2023    10
test_2025     8
dtype: int64


In [ ]:

# STEP 15 - Export corrected Hybrid (10 series, FIT/FULL protocol)

from pathlib import Path
from google.colab import files
import pandas as pd
import numpy as np

output_dir = Path("/content/hybrid_outputs")
output_dir.mkdir(exist_ok=True)

# 1. Standardize the Hybrid prediction format
hybrid_export = hybrid_corrected[
    ["timestamp_utc", "zone", "load_type",
     "energy_wh_model", "pred_hybrid_corrected"]
].copy()

hybrid_export = hybrid_export.rename(columns={
    "energy_wh_model": "y_true",
    "pred_hybrid_corrected": "y_pred"
})

hybrid_export["timestamp_utc"] = pd.to_datetime(
    hybrid_export["timestamp_utc"], utc=True
)

hybrid_export["model"] = "hybrid_leakfree"

# 2. Assign official evaluation periods
ts = hybrid_export["timestamp_utc"]

conditions = [
    (ts >= pd.Timestamp("2022-07-01", tz="UTC")) &
    (ts < pd.Timestamp("2023-01-01", tz="UTC")),

    (ts >= pd.Timestamp("2023-01-01", tz="UTC")) &
    (ts < pd.Timestamp("2023-03-01", tz="UTC")),

    (ts >= pd.Timestamp("2025-08-18", tz="UTC")) &
    (ts < pd.Timestamp("2025-10-14", tz="UTC"))
]

hybrid_export["split"] = np.select(
    conditions,
    ["validation", "test_2023", "test_2025"],
    default="other"
)

hybrid_export = hybrid_export[
    hybrid_export["split"] != "other"
].copy()

# 3. Check output integrity
keys = ["timestamp_utc", "zone", "load_type"]

assert not hybrid_export.duplicated(keys).any()
assert hybrid_export[["y_true", "y_pred"]].notna().all().all()
assert hybrid_export.groupby(
    ["zone", "load_type"]
).ngroups == 10

# 4. Prepare UTC timestamps for T38
hybrid_export["timestamp_utc"] = (
    hybrid_export["timestamp_utc"].dt.tz_convert(None)
)

hybrid_export = hybrid_export[
    ["timestamp_utc", "zone", "load_type",
     "model", "split", "y_true", "y_pred"]
].sort_values(["zone", "load_type", "timestamp_utc"])

# 5. Save three separate outputs
pred_path = output_dir / "Hybrid_Corrected_Predictions_10series.csv"
weights_path = output_dir / "Hybrid_Corrected_Weights_10series.csv"
comparison_path = output_dir / "Hybrid_Corrected_Comparison_10series.csv"

hybrid_export.to_csv(
    pred_path,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

weights_df.to_csv(weights_path, index=False)

comparison_export = pd.concat([
    hybrid_results.assign(period="SEL"),
    test_results_df
], ignore_index=True)

comparison_export.to_csv(comparison_path, index=False)

print("HYBRID EXPORT COMPLETED")
print("Total predictions:", len(hybrid_export))
print("Series:", hybrid_export.groupby(
    ["zone", "load_type"]
).ngroups)

print("\nPredictions by period:")
print(hybrid_export["split"].value_counts())

print("\nFiles saved:")
for path in [pred_path, weights_path, comparison_path]:
    print("-", path.name)
    files.download(str(path))


HYBRID EXPORT COMPLETED
Total predictions: 46007
Series: 10

Predictions by period:
split
validation    29524
test_2023      9394
test_2025      7089
Name: count, dtype: int64

Files saved:
- Hybrid_Corrected_Predictions_10series.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

- Hybrid_Corrected_Weights_10series.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

- Hybrid_Corrected_Comparison_10series.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# STEP 16 - Verify which weights were applied

validation_mask = hybrid_corrected[
    "hybrid_period"
].isin(["FIT", "SEL"])

test_mask = (
    (hybrid_corrected["timestamp_utc"] >=
     pd.Timestamp("2023-01-01", tz="UTC"))
    &
    (hybrid_corrected["timestamp_utc"] <
     pd.Timestamp("2023-03-01", tz="UTC"))
) | (
    (hybrid_corrected["timestamp_utc"] >=
     pd.Timestamp("2025-08-18", tz="UTC"))
    &
    (hybrid_corrected["timestamp_utc"] <
     pd.Timestamp("2025-10-14", tz="UTC"))
)

assert np.allclose(
    hybrid_corrected.loc[
        validation_mask, "pred_hybrid_corrected"
    ],
    hybrid_corrected.loc[
        validation_mask, "pred_hybrid_fit"
    ]
)

assert np.allclose(
    hybrid_corrected.loc[
        test_mask, "pred_hybrid_corrected"
    ],
    hybrid_corrected.loc[
        test_mask, "pred_hybrid_full"
    ]
)

print("Weight protocol verification PASSED")
print("Validation uses FIT weights")
print("Tests use FULL-validation weights")


Weight protocol verification PASSED
Validation uses FIT weights
Tests use FULL-validation weights


## 5. Results and Discussion

### 5.1 Objective and Methodology

This notebook re-evaluates the original T39 Hybrid model using the corrected GBM predictions produced in Notebook 02.

The analysis covers ten energy consumption series and preserves the original linear stacking approach, combining Chronos-Bolt and corrected GBM predictions with learned weights and an intercept.

Two sets of weights were estimated independently for each series:

- **FIT weights:** estimated using July 1 – October 15, 2022, and evaluated on the independent SEL period.
- **Full-validation weights (`w_full`):** re-estimated using July – December 2022 and applied to the 2023 and 2025 test periods.

The weight-estimation procedure uses no test observations.

### 5.2 Model Comparison on SEL

The evaluation compares Chronos, corrected GBM and retrained Hybrid predictions on common timestamps from October 16 to December 31, 2022.

| Model | Series with lowest CV(RMSE) |
|---|---:|
| Chronos | 8 |
| Corrected GBM | 0 |
| Corrected Hybrid | 2 |

The corrected Hybrid achieved the lowest CV(RMSE) in two series:

| Energy series | Chronos CV(RMSE) | Corrected Hybrid CV(RMSE) |
|---|---:|---:|
| NE Plugs & Sockets | 42.89% | 41.91% |
| NW Lighting | 66.14% | 64.39% |

Chronos achieved the lowest error in the remaining eight series.

These results describe the best model among the three evaluated candidates. They do not replace the complete T38 model-selection process, which also includes other forecasting models.

### 5.3 Out-of-Sample Test Results

After completing the SEL comparison, Hybrid weights were re-estimated using the full validation period and applied to the test periods.

The diagnostic results were:

| Test period | Chronos | Corrected GBM | Corrected Hybrid | Evaluated series |
|---|---:|---:|---:|---:|
| 2023 | 6 | 0 | 4 | 10 |
| 2025 | 5 | 1 | 2 | 8 |

These figures show which candidate achieved the lowest CV(RMSE) on each test series.

**They are diagnostic results, not model-selection results.** Final model selection must be based on the designated SEL period rather than on test performance.

Only eight series had matched predictions in the 2025 test period.

### 5.4 Main Findings

The re-evaluation shows that the corrected Hybrid can still outperform its individual components on selected energy consumption series.

However, Chronos achieved the lowest selection-period error in eight of the ten evaluated series.

The new test results differ from the earlier six-series experiment because the analysis now covers more series and uses weights re-estimated on the full validation period.

These results support the value of reviewing the original Hybrid after correcting the GBM input features, rather than assuming the original Hybrid performance remains unchanged.

### 5.5 Limitations

The analysis includes ten energy consumption series with corrected GBM forecasts, rather than all fourteen series considered in the original T38 selection.

The corrected GBM is based on a fixed 24-hour forecasting-horizon experiment. Complete operational availability of its features at a single day-ahead forecast issuance time has not yet been demonstrated.

Historical feature requirements reduce prediction coverage. Comparisons were therefore performed on timestamps available for both Chronos and the corrected GBM.

The analysis does not establish that every possible form of feature leakage has been removed.

Differences in training samples and feature configurations can also affect forecasting performance.

The 2025 test comparison includes only eight series, and small numerical improvements should not be interpreted as statistically significant without further analysis.
### Instability of Unconstrained Hybrid Weights

The Hybrid uses unconstrained ordinary least squares (OLS), allowing negative weights and coefficients that do not sum to one.

Some coefficients change substantially between FIT and full-validation estimation. For example, in NE HVAC, the Chronos weight changes from approximately 0.74 to 1.36, while the GBM weight changes from +0.25 to −0.56. In SE Plugs & Sockets, the intercept changes from approximately +13.7 to −76.9.

Such instability is associated with weaker performance on some unseen test periods. For example, in the 2025 test, the corrected Hybrid achieves approximately 62.5% CV(RMSE) for SE Plugs & Sockets, compared with 29.7% for Chronos. For SE Lighting, the corresponding errors are approximately 500% and 427%.

The original T39 methodology has been preserved in this re-evaluation. However, constrained stacking weights, such as non-negative coefficients summing to one, could be investigated in future work to improve stability. This is consistent with the project's roadmap for weighted Hybrid models.
### Differences in Evaluation Coverage

The corrected GBM scores reported in this notebook may differ slightly from those in Notebook 02 because the evaluation samples are different.

For example, NE Plugs & Sockets achieves approximately 76.21% CV(RMSE) on 1,810 SEL hours in this Hybrid re-evaluation, compared with 77.19% on 1,790 SEL hours in Notebook 02.

Notebook 02 uses timestamps common to the original and corrected GBM, whereas Notebook 03 uses timestamps common to Chronos and the corrected GBM.

Therefore, the differences reflect the evaluation coverage and do not indicate inconsistent model predictions.

### 5.6 Final Conclusion

The experimental Hybrid re-evaluation confirms that learned combinations of Chronos and corrected GBM predictions can remain beneficial for specific energy series.

On SEL, the corrected Hybrid outperformed both individual components in two of ten series, while Chronos performed best in the remaining eight.

The original FIT/SEL/full-validation weight-estimation procedure was implemented, preserving the independence of the model-selection period from weight estimation and avoiding the use of test observations for training.

These findings do not establish the final winning models across the project.

### 5.7 Next Step

The corrected Hybrid forecasts will be included in `04_Final_Model_Selection_Audit.ipynb`, together with the corrected GBM predictions and other available forecasting candidates.

The final audit will examine T38 model selection on consistent timestamps, the impact of changes in prediction coverage, and limitations relating to other candidate models that have not yet been rerun without potentially unavailable same-hour sensor features.

**Output files:**

- `Hybrid_Corrected_Predictions_10series.csv`
- `Hybrid_Corrected_Weights_10series.csv`
- `Hybrid_Corrected_Comparison_10series.csv`

**References:** `hybrid_blend_learned_weights.ipynb`, `02_GBM_Leakage_Correction.ipynb`.